# Транзакции по дням: Impala готовит, Spark читает

Как в DAG: Impala собирает свою временную таблицу, затем перезаписывает
историю по партиции дня. Spark открывает уже готовую таблицу и
`ods_alpha.scd1_trx` не читает. В DRP ничего не пишется.

Ячейки идут сверху вниз. Логин и пароль спросят во второй ячейке.
Дни июля появятся до запуска Spark. Если YARN зависнет, таблица уже готова.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
for _name in ("py4j", "py4j.java_gateway", "pyspark"):
    logging.getLogger(_name).setLevel(logging.ERROR)

REPORT_MONTH = "2026-07"
print("month", REPORT_MONTH)

In [ ]:
import getpass
import os

from rail_connectors.connection import connect

keytab_path = "/home/jovyan/test_requests/tech.keytab"
os.makedirs(os.path.dirname(keytab_path), exist_ok=True)

LAKE_USER = input("Логин Impala: ").strip()
LAKE_PASSWORD = getpass.getpass("Пароль Impala: ")
if not LAKE_USER or not LAKE_PASSWORD:
    raise RuntimeError("Логин и пароль нужны оба")

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": keytab_path,
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": LAKE_USER, "password": LAKE_PASSWORD},
)
imp._init_connection()
with imp:
    imp.execute("SELECT 1")
print("Impala ok")

## Июль по дням в свои таблицы

Один проход Impala по `ods_alpha.scd1_trx`. Фильтры секции операций:
SA, S01, не возврат, есть терминал.

`sandbox_ai.tmp_shestopalov_acq_trx_day_temp` собирается заново.
`sandbox_ai.tmp_shestopalov_acq_trx_day` — история, партиция `trx_date`.
В неё пишется так же, как `INSERT OVERWRITE ... PARTITION` в DAG.

In [ ]:
from IPython.display import display

year_i, month_i = (int(part) for part in REPORT_MONTH.split("-"))
month_start = f"{REPORT_MONTH}-01"
if month_i == 12:
    month_end = f"{year_i + 1}-01-01"
else:
    month_end = f"{year_i}-{month_i + 1:02d}-01"

temp_table = "sandbox_ai.tmp_shestopalov_acq_trx_day_temp"
hist_table = "sandbox_ai.tmp_shestopalov_acq_trx_day"

imp._init_connection()
with imp:
    imp.execute("set MEM_LIMIT=8g")
    imp.execute("set hive.exec.dynamic.partition.mode=nonstrict")
    imp.execute(f"DROP TABLE IF EXISTS {temp_table}")
    imp.execute(f"""
CREATE TABLE {temp_table} STORED AS PARQUET AS
SELECT
  cast(to_date(to_timestamp(t.d_trx_orig, 'yyyy-MM-dd HH:mm:ss')) AS string) AS trx_date,
  count(*) AS trx_cnt,
  sum(cast(t.n_amt_src AS double)) AS trx_sum,
  sum(cast(ti.n_amt_fee AS double)) AS int_component
FROM ods_alpha.scd1_trx t
LEFT JOIN ods_alpha.scd1_trx_int ti ON ti.n_trx = t.n_trx
WHERE t.c_trx_class = 'SA'
  AND t.c_trx_type = 'S01'
  AND coalesce(t.cf_trx_stat, '') <> 'R'
  AND t.c_nter IS NOT NULL
  AND trunc(to_date(to_timestamp(t.d_trx_orig, 'yyyy-MM-dd HH:mm:ss')), 'MM')
      = to_date('{month_start}')
GROUP BY 1
""")
    imp.execute(f"""
CREATE TABLE IF NOT EXISTS {hist_table} (
  trx_cnt BIGINT,
  trx_sum DOUBLE,
  int_component DOUBLE
)
PARTITIONED BY (trx_date STRING)
STORED AS PARQUET
""")
    imp.execute(f"""
INSERT OVERWRITE TABLE {hist_table} PARTITION (trx_date)
SELECT trx_cnt, trx_sum, int_component, trx_date
FROM {temp_table}
""")
    imp.execute(f"INVALIDATE METADATA {hist_table}")
    days = imp.fetch(f"""
SELECT trx_date, trx_cnt, trx_sum, int_component
FROM {hist_table}
WHERE trx_date >= '{month_start}' AND trx_date < '{month_end}'
ORDER BY trx_date
""")

print("temp", temp_table)
print("hist", hist_table)
display(days)
if days is None or len(days) == 0:
    raise RuntimeError("За июль не собралось ни одного дня.")
print("days ok", len(days))

## Spark читает готовую таблицу

Эта ячейка открывает `sandbox_ai.tmp_shestopalov_acq_trx_day`.
Озеро транзакций она не трогает. Если YARN не ответит, дни уже есть в ячейке выше.

In [ ]:
import os

from pyspark.sql import SparkSession

if "SPARK_K8S_CONF_DIR" in os.environ:
    os.environ["SPARK_CONF_DIR"] = os.environ["SPARK_K8S_CONF_DIR"]

log4j_path = "/tmp/acq_spark_log4j2.properties"
with open(log4j_path, "w", encoding="utf-8") as log4j_file:
    log4j_file.write(
        "\n".join([
            "status = error",
            "name = AcqSpark",
            "appender.console.type = Console",
            "appender.console.name = console",
            "appender.console.target = SYSTEM_ERR",
            "appender.console.layout.type = PatternLayout",
            "appender.console.layout.pattern = %p %c{1}: %m%n",
            "rootLogger.level = error",
            "rootLogger.appenderRef.stdout.ref = console",
        ])
    )
os.environ["SPARK_SUBMIT_OPTS"] = f"-Dlog4j2.configurationFile=file:{log4j_path}"

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("acq-trx-day")
    .master("yarn")
    .config("spark.yarn.queue", "ai")
    .config("spark.yarn.stagingDir", "hdfs:///tmp")
    .config("spark.yarn.maxAppAttempts", "1")
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.executor.instances", "1")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.submit.deployMode", "client")
    .config("spark.log.level", "ERROR")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

hist_table = "sandbox_ai.tmp_shestopalov_acq_trx_day"
spark.sql(f"REFRESH TABLE {hist_table}")
part = spark.sql(f"""
SELECT trx_date, trx_cnt, trx_sum, int_component
FROM {hist_table}
WHERE trx_date >= '{month_start}' AND trx_date < '{month_end}'
ORDER BY trx_date
""")
part.persist()
part.show(40, truncate=False)
n_days = part.count()
if n_days == 0:
    raise RuntimeError("Spark не увидел дни. Сначала выполните ячейку Impala.")
print("days ok", n_days)